# Provider Restart Test

For every provider that has a large model (≥ `CRASH_SIZE_GB` or name matches `CRASH_FRAGMENTS`),
loads it to trigger an OOM crash, then restarts the provider and verifies it recovers.

In [ ]:
from pathlib import Path
from unified_local_llm_server import LLMProviderPool, TransportError
from unified_local_llm_server.provider_registry import ProviderRegistry

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
registry = ProviderRegistry.load(ROOT / "providers.example.yaml")
server = LLMProviderPool(provider_registry=registry)
server.get_providers()

## Configuration

NameError: name 'server' is not defined

In [ ]:
providers = server.get_providers()
for provider in providers:
    a = server.restart(provider)
    break

In [2]:
print(a)

NameError: name 'a' is not defined

In [3]:
providers = server.get_providers()
for provider in providers:
    models = server.list_downloaded_models(provider)
    for model in models:
        print(f"{provider} - {model}")


llama_cpp - {'path': '/mnt/M/llms_models/llama_cpp/unsloth_gpt-oss-120b-GGUF_UD-Q8_K_XL_gpt-oss-120b-UD-Q8_K_XL-00001-of-00002.gguf', 'name': 'unsloth_gpt-oss-120b-GGUF_UD-Q8_K_XL_gpt-oss-120b-UD-Q8_K_XL', 'size_gb': 49.53, 'total_size_gb': 64.47, 'shards': 2, 'dir': '/mnt/M/llms_models/llama_cpp'}
llama_cpp - {'path': '/mnt/M/llms_models/hg_fc/hub/lmstudio-community/DeepSeek-R1-0528-Qwen3-8B-GGUF/DeepSeek-R1-0528-Qwen3-8B-Q4_K_M.gguf', 'name': 'DeepSeek-R1-0528-Qwen3-8B-Q4_K_M', 'size_gb': 5.03, 'total_size_gb': 5.03, 'shards': 1, 'dir': '/mnt/M/llms_models/hg_fc/hub/lmstudio-community/DeepSeek-R1-0528-Qwen3-8B-GGUF'}
llama_cpp - {'path': '/mnt/M/llms_models/hg_fc/hub/lmstudio-community/gemma-4-E4B-it-GGUF/gemma-4-E4B-it-Q4_K_M.gguf', 'name': 'gemma-4-E4B-it-Q4_K_M', 'size_gb': 5.34, 'total_size_gb': 5.34, 'shards': 1, 'dir': '/mnt/M/llms_models/hg_fc/hub/lmstudio-community/gemma-4-E4B-it-GGUF'}
llama_cpp - {'path': '/mnt/M/llms_models/hg_fc/hub/lmstudio-community/gpt-oss-20b-GGUF/gpt

In [ ]:
# Name fragments that identify 31B-class models to test
TARGET_FRAGMENTS = ["31b", "31B"]

# Name fragments to always skip (too slow / not OOM crashers)


# Small model per provider for post-restart verification
VERIFY_MODELS = {
    "llama_cpp": "DeepSeek-R1-0528-Qwen3-8B-Q4_K_M",
    "lm_studio":  None,
    "ollama":     "deepseek-r1:8b",
    "unsloth":    None,
}

## Provider Status

In [ ]:
statuses = {}
for name in server.get_providers():
    print(name)
    s = await server.check_provider(name)
    statuses[name] = s
    print(f"  {'OK' if s['ok'] else '--'}  {name:<12}  {s['server_url']}")

name


## Run — Load 31B Models, Detect Crashes Automatically

Loads each 31B model. If the server becomes unreachable after loading → crash confirmed → restart + verify.

In [ ]:
for provider in server.get_providers():
    if not statuses.get(provider, {}).get("ok"):
        print(f"\n[{provider}] SKIP — not reachable")
        continue

    verify_model = VERIFY_MODELS.get(provider)
    if not verify_model:
        print(f"\n[{provider}] SKIP — no verify model configured")
        continue

    models = server.list_downloaded_models(provider)
    target_models = [
        m["name"] for m in models
        if not any(f in m["name"] for f in SKIP_FRAGMENTS)
        and any(f in m["name"] for f in TARGET_FRAGMENTS)
    ]

    if not target_models:
        print(f"\n[{provider}] no 31B models found")
        continue

    print(f"\n[{provider}] found {len(target_models)} 31B model(s): {target_models}")

    for model in target_models:
        print(f"\n{'='*60}")
        print(f"  {provider} / {model}")
        print(f"{'='*60}")

        print("[1] Loading...")
        try:
            llm = server.load_model(provider, model)
            await llm.call(messages=[{"role": "user", "content": "Hi"}],
                           options={"num_predict": 5})
            print("    No crash — loaded without OOM")
            server.unload_all_models(provider)
            continue
        except TransportError as exc:
            print(f"    CRASH DETECTED: {exc}")
        except Exception as exc:
            print(f"    Model error (provider still up): {type(exc).__name__}: {exc}")
            continue

        print("[2] Restarting provider...")
        try:
            result = server.restart(provider)
            if result.get("returncode") == 0:
                print(f"    OK — back online via {result['method']} ({result.get('waited_s', 0):.1f}s)")
            else:
                print(f"    RESTART FAILED: {result.get('stderr', '')}")
                break
        except ValueError as exc:
            print(f"    Cannot restart — {exc}")
            break

        print("[3] Verifying with small model...")
        try:
            llm = server.load_model(provider, verify_model)
            response = await llm.call(
                messages=[{"role": "user", "content": "Reply with one word: working"}],
                options={"num_predict": 10},
            )
            print(f"    PASS — {response.strip()}")
        except Exception as exc:
            print(f"    FAIL — {type(exc).__name__}: {exc}")

print("\nDone.")

## Run Crash → Restart → Verify